In [1]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "id": "c94fe552",
   "metadata": {},
   "source": [
    "# Step 1 of RAG: Document Loading and Chunking Strategies\n",
    "\n",
    "RAG (Retrieval-Augmented Generation) has 4 core steps:\n",
    "1. **Document Loading and Chunking** (this notebook)\n",
    "2. Generating embeddings\n",
    "3. Storing in a vector database\n",
    "4. Retrieval and passing context to an LLM\n",
    "\n",
    "### What you will learn\n",
    "1. Why chunking matters\n",
    "2. Loading `.txt` and `.pdf` documents\n",
    "3. Five chunking strategies (fixed-size, `CharacterTextSplitter`, `RecursiveCharacterTextSplitter`, sentence-based, token-based)\n",
    "4. Comparing all strategies"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "4dada3a4",
   "metadata": {},
   "source": [
    "## Why Chunking Matters\n",
    "- Embedding models have a maximum input length (often 256 to 512 tokens).\n",
    "- One giant embedding blurs many ideas together and gives poor search results.\n",
    "- Smaller, focused chunks give more accurate embeddings and retrieval."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 1,
   "id": "81679324",
   "metadata": {},
   "outputs": [],
   "source": [
    "!pip install -q langchain-text-splitters tiktoken pypdf fpdf2 pandas matplotlib"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "4ce46567",
   "metadata": {},
   "source": [
    "## 1. Create a Sample Document\n",
    "We generate a file locally so the notebook runs without external downloads."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 1,
   "id": "5134f45f",
   "metadata": {},
   "outputs": [],
   "source": [
    "import os\n",
    "os.makedirs(\"docs\", exist_ok=True)\n",
    "\n",
    "article_text = \"\"\"Space Exploration: A Brief History\n",
    "\n",
    "Space exploration began in 1957 when the Soviet Union launched Sputnik 1, the first artificial satellite. This event started the Space Race between the United States and the Soviet Union and led to rapid advances in rocket technology.\n",
    "\n",
    "In 1969, NASA's Apollo 11 mission landed the first humans on the Moon. Neil Armstrong and Buzz Aldrin walked on the lunar surface while Michael Collins orbited above. The mission proved that humans could travel to another world and return safely.\n",
    "\n",
    "India entered the field through the Indian Space Research Organisation, or ISRO. In 2014, the Mars Orbiter Mission made India the first country to reach Mars orbit on its first attempt. In 2023, Chandrayaan-3 landed near the Moon's south pole, a region no other mission had reached.\n",
    "\n",
    "Modern space exploration also involves private companies. Reusable rockets have greatly reduced launch costs, which makes it cheaper to place satellites in orbit. These satellites provide internet access, weather forecasts, and navigation services to billions of people.\n",
    "\n",
    "Looking ahead, space agencies are planning crewed missions to Mars and permanent bases on the Moon. Scientists are also searching for signs of life on distant planets using powerful space telescopes.\n",
    "\"\"\"\n",
    "\n",
    "with open(\"docs/space_article.txt\", \"w\") as f:\n",
    "    f.write(article_text)\n",
    "\n",
    "print(\"Characters:\", len(article_text), \"| Words:\", len(article_text.split()))"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "eec59eda",
   "metadata": {},
   "source": [
    "## 2. Loading a `.txt` Document"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 1,
   "id": "e2383a05",
   "metadata": {},
   "outputs": [],
   "source": [
    "with open(\"docs/space_article.txt\", \"r\") as f:\n",
    "    loaded_text = f.read()\n",
    "\n",
    "print(loaded_text[:300], \"...\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "07b79a74",
   "metadata": {},
   "source": [
    "## 3. Loading a `.pdf` Document"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 1,
   "id": "78165c0f",
   "metadata": {},
   "outputs": [],
   "source": [
    "from fpdf import FPDF\n",
    "\n",
    "pdf = FPDF()\n",
    "pdf.add_page()\n",
    "pdf.set_font(\"Times\", size=12)\n",
    "for line in article_text.split(\"\n\"):\n",
    "    pdf.multi_cell(0, 8, line, new_x=\"LMARGIN\", new_y=\"NEXT\")\n",
    "pdf.output(\"docs/space_article.pdf\")\n",
    "print(\"Created docs/space_article.pdf\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 1,
   "id": "627f6605",
   "metadata": {},
   "outputs": [],
   "source": [
    "from pypdf import PdfReader\n",
    "\n",
    "reader = PdfReader(\"docs/space_article.pdf\")\n",
    "pdf_text = \"\n\".join(page.extract_text() for page in reader.pages)\n",
    "\n",
    "print(\"Pages:\", len(reader.pages))\n",
    "print(pdf_text[:300], \"...\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "266b6bfe",
   "metadata": {},
   "source": [
    "Loading depends on the file format, but the goal is always the same: **plain text**. From here on we chunk `loaded_text` (from the `.txt` file).\n",
    "\n",
    "## 4. Strategy 1: Fixed-Size Character Chunking (from scratch)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 1,
   "id": "e48c51e1",
   "metadata": {},
   "outputs": [],
   "source": [
    "def fixed_size_chunking(text, chunk_size=300, overlap=50):\n",
    "    chunks, start = [], 0\n",
    "    while start < len(text):\n",
    "        chunks.append(text[start:start + chunk_size])\n",
    "        start += chunk_size - overlap\n",
    "    return chunks\n",
    "\n",
    "fixed_chunks = fixed_size_chunking(loaded_text, chunk_size=300, overlap=50)\n",
    "print(\"Number of chunks:\", len(fixed_chunks))\n",
    "for i in range(2):\n",
    "    print(f\"\n--- Chunk {i+1} ({len(fixed_chunks[i])} chars) ---\n{fixed_chunks[i]}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "4f7f63be",
   "metadata": {},
   "source": [
    "Notice chunks can end in the middle of a word or sentence, since this method ignores text structure."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "27f75ee8",
   "metadata": {},
   "source": [
    "## 5. Strategy 2: `CharacterTextSplitter` (LangChain)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 1,
   "id": "9adb5d50",
   "metadata": {},
   "outputs": [],
   "source": [
    "from langchain_text_splitters import CharacterTextSplitter\n",
    "\n",
    "char_splitter = CharacterTextSplitter(separator=\"\n\n\", chunk_size=300, chunk_overlap=50)\n",
    "char_chunks = char_splitter.split_text(loaded_text)\n",
    "\n",
    "print(\"Number of chunks:\", len(char_chunks))\n",
    "for i, c in enumerate(char_chunks):\n",
    "    print(f\"\n--- Chunk {i+1} ({len(c)} chars) ---\n{c}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "1510beb8",
   "metadata": {},
   "source": [
    "This splitter only splits on the chosen separator (blank line), so a paragraph longer than `chunk_size` stays as one big chunk (LangChain may print a warning about it).\n",
    "\n",
    "## 6. Strategy 3: `RecursiveCharacterTextSplitter` (recommended default)\n",
    "It tries paragraph breaks first, then lines, then sentences, then words, and only cuts characters as a last resort."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 1,
   "id": "a38c7a1c",
   "metadata": {},
   "outputs": [],
   "source": [
    "from langchain_text_splitters import RecursiveCharacterTextSplitter\n",
    "\n",
    "recursive_splitter = RecursiveCharacterTextSplitter(\n",
    "    chunk_size=300, chunk_overlap=50,\n",
    "    separators=[\"\n\n\", \"\n\", \". \", \" \", \"\"],\n",
    ")\n",
    "recursive_chunks = recursive_splitter.split_text(loaded_text)\n",
    "\n",
    "print(\"Number of chunks:\", len(recursive_chunks))\n",
    "for i, c in enumerate(recursive_chunks):\n",
    "    print(f\"\n--- Chunk {i+1} ({len(c)} chars) ---\n{c}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "5b4f254c",
   "metadata": {},
   "source": [
    "## 7. Strategy 4: Sentence-Based Chunking"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 1,
   "id": "8336fcc7",
   "metadata": {},
   "outputs": [],
   "source": [
    "import re\n",
    "\n",
    "def sentence_chunking(text, sentences_per_chunk=2):\n",
    "    sentences = [s for s in re.split(r\"(?<=[.!?])\s+\", text.strip()) if s]\n",
    "    return [\" \".join(sentences[i:i + sentences_per_chunk])\n",
    "            for i in range(0, len(sentences), sentences_per_chunk)]\n",
    "\n",
    "sentence_chunks = sentence_chunking(loaded_text, sentences_per_chunk=2)\n",
    "print(\"Number of chunks:\", len(sentence_chunks))\n",
    "for i, c in enumerate(sentence_chunks[:3]):\n",
    "    print(f\"\n--- Chunk {i+1} ---\n{c}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "e4792950",
   "metadata": {},
   "source": [
    "## 8. Strategy 5: Token-Based Chunking\n",
    "Models limit input in **tokens** (roughly 4 characters each in English), so token chunking guarantees the limit is respected."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 1,
   "id": "392a88aa",
   "metadata": {},
   "outputs": [],
   "source": [
    "import tiktoken\n",
    "\n",
    "encoding = tiktoken.get_encoding(\"cl100k_base\")\n",
    "\n",
    "def token_chunking(text, max_tokens=60, overlap_tokens=10):\n",
    "    tokens = encoding.encode(text)\n",
    "    chunks, start = [], 0\n",
    "    while start < len(tokens):\n",
    "        chunks.append(encoding.decode(tokens[start:start + max_tokens]))\n",
    "        start += max_tokens - overlap_tokens\n",
    "    return chunks\n",
    "\n",
    "token_chunks = token_chunking(loaded_text, max_tokens=60, overlap_tokens=10)\n",
    "print(\"Number of chunks:\", len(token_chunks))\n",
    "for i, c in enumerate(token_chunks[:3]):\n",
    "    print(f\"\n--- Chunk {i+1} ({len(encoding.encode(c))} tokens) ---\n{c}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "fa0737ea",
   "metadata": {},
   "source": [
    "## 9. Compare All Strategies"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 1,
   "id": "dcc07bd4",
   "metadata": {},
   "outputs": [],
   "source": [
    "import pandas as pd\n",
    "\n",
    "strategies = {\n",
    "    \"Fixed-size (scratch)\": fixed_chunks,\n",
    "    \"CharacterTextSplitter\": char_chunks,\n",
    "    \"RecursiveCharacterTextSplitter\": recursive_chunks,\n",
    "    \"Sentence-based\": sentence_chunks,\n",
    "    \"Token-based\": token_chunks,\n",
    "}\n",
    "\n",
    "comparison = pd.DataFrame({\n",
    "    \"Strategy\": list(strategies),\n",
    "    \"Number of Chunks\": [len(c) for c in strategies.values()],\n",
    "    \"Avg Length (chars)\": [round(sum(map(len, c)) / len(c), 1) for c in strategies.values()],\n",
    "    \"Max Length (chars)\": [max(map(len, c)) for c in strategies.values()],\n",
    "})\n",
    "comparison"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 1,
   "id": "031928ca",
   "metadata": {},
   "outputs": [],
   "source": [
    "import matplotlib.pyplot as plt\n",
    "\n",
    "plt.figure(figsize=(8, 5))\n",
    "plt.bar(comparison[\"Strategy\"], comparison[\"Number of Chunks\"])\n",
    "plt.xticks(rotation=30, ha=\"right\")\n",
    "plt.ylabel(\"Number of chunks\")\n",
    "plt.title(\"Chunks produced by each strategy\")\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "2eeb3d8a",
   "metadata": {},
   "source": [
    "## 10. Practice Exercises (Solved)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 1,
   "id": "45db7c61",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Exercise 1: effect of chunk_size on RecursiveCharacterTextSplitter\n",
    "for size in [600, 300, 150]:\n",
    "    s = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=size // 6)\n",
    "    print(f\"chunk_size={size:4d} -> {len(s.split_text(loaded_text))} chunks\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 1,
   "id": "6d9cb0af",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Exercise 2: sentences_per_chunk = 1, 2, 4\n",
    "for n in [1, 2, 4]:\n",
    "    print(f\"sentences_per_chunk={n} -> {len(sentence_chunking(loaded_text, n))} chunks\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 1,
   "id": "d7d4f85c",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Exercise 3: run all 5 strategies on the text extracted from the PDF\n",
    "pdf_strategies = {\n",
    "    \"Fixed-size\": fixed_size_chunking(pdf_text),\n",
    "    \"Character\": char_splitter.split_text(pdf_text),\n",
    "    \"Recursive\": recursive_splitter.split_text(pdf_text),\n",
    "    \"Sentence\": sentence_chunking(pdf_text),\n",
    "    \"Token\": token_chunking(pdf_text),\n",
    "}\n",
    "for name, chunks in pdf_strategies.items():\n",
    "    print(f\"{name:12s} -> {len(chunks)} chunks\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "4090c51b",
   "metadata": {},
   "source": [
    "`CharacterTextSplitter` returns very few chunks here because PDF extraction usually separates paragraphs with single line breaks, not the blank line (`\n\n`) the splitter looks for. `RecursiveCharacterTextSplitter` still works because it falls back to other separators. This is a good reason to prefer it."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "1043e609",
   "metadata": {},
   "source": [
    "## Key Takeaways\n",
    "\n",
    "| Strategy | Best for | Watch out for |\n",
    "|---|---|---|\n",
    "| Fixed-size | Learning the core idea | Cuts words and sentences |\n",
    "| `CharacterTextSplitter` | Simple separator-based splitting | Rigid, big paragraphs stay big |\n",
    "| `RecursiveCharacterTextSplitter` | Most RAG use cases | Slightly more setup |\n",
    "| Sentence-based | Precision-sensitive text | Uneven chunk sizes |\n",
    "| Token-based | Respecting model token limits | Needs a tokenizer |\n",
    "\n",
    "`chunk_overlap` repeats a little text between neighbouring chunks so ideas on a boundary are not lost.\n",
    "The next RAG step is turning each chunk into an embedding and storing it in a vector database."
   ]
  }
 ],
 "metadata": {
  "colab": {
   "provenance": []
  },
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}

<>:257: SyntaxWarning: invalid escape sequence '\s'
<>:257: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_2355/952534197.py:257: SyntaxWarning: invalid escape sequence '\s'
  "    sentences = [s for s in re.split(r\"(?<=[.!?])\s+\", text.strip()) if s]\n",


{'cells': [{'cell_type': 'markdown',
   'id': 'c94fe552',
   'metadata': {},
   'source': ['# Step 1 of RAG: Document Loading and Chunking Strategies\n',
    '\n',
    'RAG (Retrieval-Augmented Generation) has 4 core steps:\n',
    '1. **Document Loading and Chunking** (this notebook)\n',
    '2. Generating embeddings\n',
    '3. Storing in a vector database\n',
    '4. Retrieval and passing context to an LLM\n',
    '\n',
    '### What you will learn\n',
    '1. Why chunking matters\n',
    '2. Loading `.txt` and `.pdf` documents\n',
    '3. Five chunking strategies (fixed-size, `CharacterTextSplitter`, `RecursiveCharacterTextSplitter`, sentence-based, token-based)\n',
    '4. Comparing all strategies']},
  {'cell_type': 'markdown',
   'id': '4dada3a4',
   'metadata': {},
   'source': ['## Why Chunking Matters\n',
    '- Embedding models have a maximum input length (often 256 to 512 tokens).\n',
    '- One giant embedding blurs many ideas together and gives poor search results.\n',
  